In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "bohn2015communication")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Bohn_2015_tab_ApeData_Data_AbsentEntities.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:

import pandas as pd
import numpy as np
df = pd.read_csv(complete_path_1)


df['study_id']="bohn2015communication"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)

df = df.rename(columns={"subject": "ape"})

In [3]:
df[['group','group1']] = df['species'].str.split('-',expand=True)

group_replace = ['orang', 'gorilla', 'bonobo']
for x in group_replace:
    df['group'] = df['group'].str.replace(x, '')
    # df['group'] = df['group'].str.replace('gorilla', '')
    # df['group'] = df['group'].str.replace('bonobo', '')

# print(df['group'])

In [4]:
df['ape'] = df['ape'].str.rstrip()

comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)

for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)  
df= df.merge(apedf,left_on='ape', right_on='name', how='left')


In [5]:

df = df.rename(columns={"sex_y": "sex",
    "species_y": "species",
    "test trial": "test_trial",
    "common ground": "common_ground",
    "food left": "food_left",
    "food right": "food_right",
    "side indicated": "side_indicated",
    "food indicated": "food_indicated",
    "point to absent": "point_to_absent",
    "indicated food": "indicated_food",
    "point to absent hq food": "point_to_absent_hq_food",
    "point to absent lq food": "point_to_absent_lq_food",
    "point to middle before": "point_to_middle_before",
    "nr. of points": "nr_of_points",
    "prompt?": "prompt"})


In [6]:
df.rename(columns={"ape": "participant", "group":"species_subgroup"}, inplace=True)
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 

df= df.merge(ape_dob,left_on='participant', right_on='name', how='left') #insert dob of participants
df['dodc'] = df['year'].astype(str) + '-' + df['month'].astype(str) + '-' + df['day'].astype(str)
df['dodc'] = pd.to_datetime(df['dodc'])##convert date of data collection to datetime format
df['dob'] = pd.to_datetime(df['dob'])##convert date of birth to datetime format

df['age_in_years'] = (df['dodc'] - df['dob']).dt.days//365

In [7]:
# df.columns

In [8]:
df = df[['study_id', 'experiment','year', 'month', 'day', 'participant','age_in_years' ,
        'sex', 'species', 'species_subgroup', 'session', 'test_trial', 'trial', 'condition', 
       'constellation', 'food_left',
       'food_right', 'point', 'side_indicated', 'food_indicated',
       'point_to_absent', 'indicated_food', 
       'prompt' ]]

In [9]:
# exp1 = df[df['experiment'] == 1]
# exp1 = exp1.dropna(axis=1, how='all')

for index in range(1,3):
    exp = df[df['experiment'] == index]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'bohn2015communication_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'bohn2015communication_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)